In [1]:
import torch
from torch import nn
from transformers import BartTokenizer, BartForConditionalGeneration, Trainer, TrainingArguments, BartConfig, AdamW
from tqdm.notebook import tqdm
import pandas as pd
from datasets import Dataset, load_metric, DatasetDict
from torch.nn.init import xavier_uniform_
import torch.nn.functional as F
from transformers import AdamW, get_linear_schedule_with_warmup, BartForSequenceClassification

       
model_name = 'facebook/bart-base'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
'''tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)
print ("device ",device)
model = model.to(device)'''
mapping = {"N":"negation", "E":"entity swap", "#":"number swap", 
           "A":"to antonym", "I":"no information", 
           "X":"mutual exclusion"}
mapping2 = {"N":0, "E":1, "#":2, 
           "A":3, "I":4, 
           "X":5, "Ans":6}
mapping3 = {0:"N", 1:"E", 2:"#", 
           3:"A", 4:"I", 
           5:"X", 6:"Ans"}  

model_name = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(model_name)
config = BartConfig.from_pretrained(model_name)


class MTLModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Initialize BART model for generation
        self.bart = BartForConditionalGeneration.from_pretrained("facebook/bart-base")

        # Initialize classification heads
        self.classifier_1 = nn.Linear(config.d_model, 2)
        self.classifier_2 = nn.Linear(config.d_model, len(mapping2))

        self.dropout = nn.Dropout(0.2)
        self.criterion = nn.CrossEntropyLoss()

        self.initial_weights = [1, 0.2]  # Weights for generation and classification losses
        self.loss_weights = self.initial_weights.copy()
        self.optimizer = None
        self.scheduler = None
        self.init_params()

    def init_params(self):
        # Initialize weights for classification heads
        for m in [self.classifier_1, self.classifier_2]:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, input_ids, attention_mask, labels=None, 
            classification_input_ids_task_1=None, classification_attention_mask_task_1=None, 
            classification_labels_task_1=None, 
            classification_input_ids_task_2=None, classification_attention_mask_task_2=None, 
            classification_labels_task_2=None):
        # Generation outputs
        generation_outputs = self.bart(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            labels=labels, 
            return_dict=True,
            output_hidden_states=True
        )

        # Classification outputs for task 1
        classification_outputs_1 = self.bart(
            input_ids=classification_input_ids_task_1,
            attention_mask=classification_attention_mask_task_1,
            return_dict=True,
            output_hidden_states=True
        )

        # Classification outputs for task 2
        classification_outputs_2 = self.bart(
            input_ids=classification_input_ids_task_2,
            attention_mask=classification_attention_mask_task_2,
            return_dict=True,
            output_hidden_states=True
        )

        # Extract hidden states from the classification model
        hidden_states_1 = classification_outputs_1.decoder_hidden_states[-1].detach()
        hidden_states_2 = classification_outputs_2.decoder_hidden_states[-1].detach()

        # Get EOS token hidden states for both tasks
        eos_token_id = self.bart.config.eos_token_id
        eos_mask_1 = classification_input_ids_task_1.eq(eos_token_id).detach()
        eos_mask_2 = classification_input_ids_task_2.eq(eos_token_id).detach()

        eos_hidden_state_1 = hidden_states_1[eos_mask_1].view(hidden_states_1.size(0), -1, hidden_states_1.size(-1))[:, -1, :]
        eos_hidden_state_2 = hidden_states_2[eos_mask_2].view(hidden_states_2.size(0), -1, hidden_states_2.size(-1))[:, -1, :]

        eos_hidden_state_1 = self.dropout(eos_hidden_state_1.float())
        eos_hidden_state_2 = self.dropout(eos_hidden_state_2.float())

        # Classification heads for both tasks
        logits_1 = self.classifier_1(eos_hidden_state_1)
        logits_2 = self.classifier_2(eos_hidden_state_2)

        if self.training:
            # Ensure classification labels have the correct shape and type
            if classification_labels_task_1 is not None:
                classification_labels_task_1 = classification_labels_task_1.long()

            if classification_labels_task_2 is not None:
                classification_labels_task_2 = classification_labels_task_2.long()

            # Calculate losses
            loss_g = generation_outputs.loss
            loss_r1 = self.criterion(logits_1, classification_labels_task_1) if classification_labels_task_1 is not None else torch.tensor(0.0)
            loss_r2 = self.criterion(logits_2, classification_labels_task_2) if classification_labels_task_2 is not None else torch.tensor(0.0)

            weighted_loss_g = self.loss_weights[0] * loss_g
            weighted_loss_r1 = self.loss_weights[1] * loss_r1
            weighted_loss_r2 = self.loss_weights[1] * loss_r2

            total_loss = weighted_loss_g + weighted_loss_r1 + weighted_loss_r2

            return total_loss, logits_1, logits_2, loss_g

        # Modify input for generation based on classification outputs
        predicted_class_1 = logits_1.argmax(dim=-1).detach()
        predicted_class_2 = logits_2.argmax(dim=-1).detach()
        modified_decoder_input_ids = self.modify_decoder_input(input_ids, predicted_class_1, predicted_class_2)

        # Generate text using modified input
        generation_outputs = self.bart.generate(
            input_ids=modified_decoder_input_ids, 
            attention_mask=attention_mask
        )

        return generation_outputs, logits_1, logits_2


    def modify_decoder_input(self, input_ids, predicted_class_1, predicted_class_2):
        # Combine classification results to modify the input
        predicted_class_1_ids = predicted_class_1.unsqueeze(1)
        predicted_class_2_ids = predicted_class_2.unsqueeze(1)
        combined_class_ids = torch.cat([predicted_class_1_ids, predicted_class_2_ids], dim=1)
        modified_input_ids = torch.cat([combined_class_ids, input_ids[:, 2:]], dim=1)
        return modified_input_ids

    def update_loss_weights(self):
        # Gradually shift weights towards generation
        gen_weight = self.initial_weights[0] * 1.03
        class_weight = self.initial_weights[1] * 0.97
        self.loss_weights = [gen_weight, class_weight]

    def configure_optimizers(self, learning_rate=5e-5, weight_decay=0.01, adam_epsilon=1e-8, warmup_steps=0, total_steps=0):
        no_decay = ["bias", "LayerNorm.weight"]
        optimizer_grouped_parameters = [
            {
                "params": [
                    p for n, p in self.named_parameters()
                    if not any(nd in n for nd in no_decay)
                ],
                "weight_decay": weight_decay
            },
            {
                "params": [
                    p for n, p in self.named_parameters()
                    if any(nd in n for nd in no_decay)
                ],
                "weight_decay": 0.0
            }
        ]
        optimizer = AdamW(
            optimizer_grouped_parameters,
            lr=learning_rate,
            eps=adam_epsilon
        )
        self.optimizer = optimizer
        
        if warmup_steps > 0 and total_steps > 0:
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps
            )
        else:
            scheduler = None
        self.scheduler = scheduler
        
        return optimizer, scheduler

    def optimizer_step(self, epoch, batch_idx):
        if self.optimizer is not None:
            self.optimizer.step()
            self.optimizer.zero_grad()
            if self.scheduler is not None:
                self.scheduler.step()


#model = MTLModel.from_pretrained('./fine-tuned-bartMTL-salience') replace these names with the checkpoint....
#model.load_state_dict(torch.load('./fine-tuned-bartMTL-salience/pytorch_model.bin'))
model = MTLModel(config).to(device)

2024-08-20 10:39:30.546933: I external/local_tsl/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2024-08-20 10:39:30.582550: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-20 10:39:31.345756: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/hmoradis/.conda/envs/QA_env/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(

In [2]:
def answerability(context, question):
    st = f"""Context: {context}

    Question: {question}"""     
    return st

def mission(context, question, label, sal):
    if label != "Ans":
        st = f"""Context: {context}

        Question: {question}

        Issue: {mapping[label]}

        Salient Sentence: {sal}

        Task: Generate a new question based on the context and salient sentence that can be answered with the given context, given the issue."""
    else:
        st = question
    return st

def labelLess(context, question, sal, ab):
    if ab == 0:
        st = f"""Context: {context}

        Question: {question}

        Salient Sentence: {sal}
        """
    else:
        st = question
    return st



def tokenize_function(examples):
    # Prepare inputs for Task 1

    # Prepare inputs for Task 2, which depends on the output of Task 1
    classification_inputs_task_1 = [
        answerability(context, question)  # Assuming you have a function to process Task 2 inputs
        for context, question in zip(examples['context'], examples['potential_UQ_question'])
    ]
    classification_targets_task_1 = examples["label_indicator"]
    
    
    classification_inputs_task_2 = [
        labelLess(context if label != "I" else noinfoc, question if label != "I" else noinfoq, sal, ab)
        for context, question, label, sal, noinfoc, noinfoq, ab in zip(examples['context'], examples['potential_UQ_question'], examples['label'], examples['answer_sentence'], examples['old_context'], examples['original_question'], examples['label_indicator'])
    ]
    classification_targets_task_2 = [
        mapping2[label] for label in examples['label']
    ]

    # Prepare inputs for Task 3 (Text Generation), dependent on Task 2
    inputs_task_3 = [
        mission(context if label != "I" else noinfoc, question if label != "I" else noinfoq, label, sal)
        for context, question, label, sal, noinfoc, noinfoq in zip(examples['context'], examples['potential_UQ_question'], examples['label'], examples['answer_sentence'], examples['old_context'], examples['original_question'])
    ]
    targets_task_3 = [
        original_q if label != "I" else potential_q 
        for label, original_q, potential_q in zip(examples['label'], examples['original_question'], examples['potential_UQ_question'])
    ]
    
    # Tokenize inputs for all tasks
    model_inputs_task_1 = tokenizer(classification_inputs_task_1, max_length=512, truncation=True, padding='max_length')
    model_inputs_task_2 = tokenizer(classification_inputs_task_2, max_length=512, truncation=True, padding='max_length')
    model_inputs_task_3 = tokenizer(inputs_task_3, max_length=512, truncation=True, padding='max_length')
    
    # Tokenize targets for text generation (Task 3)
    with tokenizer.as_target_tokenizer():
        labels_task_3 = tokenizer(targets_task_3, max_length=128, truncation=True, padding='max_length')

    # Ensure the labels are correctly formatted
    labels_ids_task_3 = labels_task_3['input_ids']
    
    return {
        'input_ids': model_inputs_task_3['input_ids'],  # Use the task 3 inputs
        'attention_mask': model_inputs_task_3['attention_mask'],
        'labels': labels_ids_task_3,
        'classification_input_ids_task_1': model_inputs_task_1['input_ids'],
        'classification_attention_mask_task_1': model_inputs_task_1['attention_mask'],
        'classification_labels_task_1': classification_targets_task_1,
        'classification_input_ids_task_2': model_inputs_task_2['input_ids'],
        'classification_attention_mask_task_2': model_inputs_task_2['attention_mask'],
        'classification_labels_task_2': classification_targets_task_2
    }


In [3]:
csv_file = 'all_unans_salience.csv'  # Replace with your CSV file path

df = pd.read_csv(csv_file)
df['label_indicator'] = df['label_indicator'].astype(int)
'''column_of_interest = 'label'
value_of_interest = 'Ans'

# Filter rows where the column has the specified value
filtered_rows = df[df[column_of_interest] == value_of_interest]

# Calculate the number of rows to keep (25% of the total)
num_rows_to_keep = int(len(filtered_rows) * 0.1)

# Sample the rows to keep
sampled_rows = filtered_rows.sample(n=num_rows_to_keep, random_state=1)

# Drop the rows with the specified value
df = df[df[column_of_interest] != value_of_interest]

# Concatenate the sampled rows back to the DataFrame
df = pd.concat([df, sampled_rows])'''

# Reset index
df = df.reset_index(drop=True)
df

,Unnamed: 0,context,original_question,potential_UQ_question,original_answer_span,answer_sentence,label,old_context,label_indicator
0,0,"In 1609, while still there, Smyth wrote a trac...",Smyth believed a scriptural church should cons...,Smyth believed a scriptural church should cons...,baptized on a personal confession of faith,"In 1609, while still there, Smyth wrote a trac...",A,NaN,0
1,1,"In 1609, while still there, Smyth wrote a trac...",Smyth believed a scriptural church should cons...,Smyth believed a scriptural church should cons...,baptized on a personal confession of faith,"In 1609, while still there, Smyth wrote a trac...",A,NaN,0
2,2,"In 1609, while still there, Smyth wrote a trac...",Smyth believed a scriptural church should cons...,Smyth believed a scriptural church should cons...,baptized on a personal confession of faith,"In 1609, while still there, Smyth wrote a trac...",A,NaN,0
3,3,"In October 2015, TCM announced the launch of t...",How long do TCM Wineclub subscriptions last?,How unretentive do TCM Wineclub subscriptions ...,3 month,"Wines are available in 3 month subscriptions, ...",A,NaN,0
4,4,Some Christian writers considered the possibil...,As what was the event mistaken by some pagans?,As what was the event mistaken by no pagans?,a solar eclipse,Some Christian writers considered the possibil...,A,NaN,0
...,...,...,...,...,...,...,...,...,...
20471,20471,Most residents belong to the Anglican Communio...,The Diocese of Saint Helena has it's own what?,The Diocese of Saint Helena has it's own what?,bishop,Most residents belong to the Anglican Communio...,Ans,NaN,1
20472,20472,Other Christian denominations on the island in...,What year did the Salvation Army show up on Sa...,What year did the Salvation Army show up on Sa...,1884,Other Christian denominations on the island in...,Ans,NaN,1
20473,20473,Other Christian denominations on the island in...,What year did the Salvation Army show up on Sa...,What year did the Salvation Army show up on Sa...,1884,Other Christian denominations on the island in...,Ans,NaN,1
20474,20474,Other Christian denominations on the island in...,When did Baptists come to the island?,When did Baptists come to the island?,1845,Other Christian denominations on the island in...,Ans,NaN,1


In [4]:
# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)
# Split the dataset into train and validation sets
dataset = dataset.train_test_split(test_size=0.2)
dataset = DatasetDict({"train": dataset["train"], "validation": dataset["test"]})
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/16380 [00:00<?, ? examples/s]

/home/hmoradis/.conda/envs/QA_env/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:4126: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/4096 [00:00<?, ? examples/s]

In [5]:
def filter_dict(example):
    return {
        'input_ids': example['input_ids'],  # Task 3 inputs
        'attention_mask': example['attention_mask'],
        'labels': example['labels'],
        'classification_input_ids_task_1': example['classification_input_ids_task_1'],
        'classification_attention_mask_task_1': example['classification_attention_mask_task_1'],
        'classification_labels_task_1': example['classification_labels_task_1'],
        'classification_input_ids_task_2': example['classification_input_ids_task_2'],
        'classification_attention_mask_task_2': example['classification_attention_mask_task_2'],
        'classification_labels_task_2': example['classification_labels_task_2']
    }


filtered_dataset_dict = {}
for split, dataset in tokenized_datasets.items():
    filtered_dataset_dict[split] = dataset.map(
        filter_dict, 
        remove_columns=[col for col in dataset.column_names if col not in [
            'input_ids', 
            'attention_mask', 
            'labels',
            'classification_input_ids_task_1',
            'classification_attention_mask_task_1', 
            'classification_labels_task_1',
            'classification_input_ids_task_2',
            'classification_attention_mask_task_2', 
            'classification_labels_task_2'
        ]]
    )

Map:   0%|          | 0/16380 [00:00<?, ? examples/s]

Map:   0%|          | 0/4096 [00:00<?, ? examples/s]

In [6]:
train = filtered_dataset_dict['train']
validation = filtered_dataset_dict['validation']

In [7]:
for k, v in train[0].items():
    print(k)

input_ids
attention_mask
labels
classification_input_ids_task_1
classification_attention_mask_task_1
classification_labels_task_1
classification_input_ids_task_2
classification_attention_mask_task_2
classification_labels_task_2


In [9]:
#from safetensors.torch import load_file
'''output_dir = './checkpoint-44862'
#output_dir = './fine-tuned-bartMTL-salience'

# Load the configuration
config = BartConfig.from_pretrained(output_dir)

# Reconstruct the model
model = MTLModel(config)
model.bart.model.encoder.load_state_dict(load_file(os.path.join(output_dir, "encoder.safetensors")))
model.bart.model.decoder.load_state_dict(load_file(os.path.join(output_dir, "decoder.safetensors")))
model.bart.model.shared.load_state_dict(load_file(os.path.join(output_dir, "shared.safetensors")))
model.bart.lm_head.load_state_dict(load_file(os.path.join(output_dir, "lm_head.safetensors")))
model.classifier.load_state_dict(load_file(os.path.join(output_dir, "classifier.safetensors")))

# Load the tokenizer
tokenizer = BartTokenizer.from_pretrained(output_dir)

# Now you can use `model` and `tokenizer` as needed
model.to(device)'''
from safetensors.torch import load_file
import os
output_dir = './fine-tuned-bartMTL-salience'

# Load the configuration
config = BartConfig.from_pretrained(output_dir)

# Reconstruct the model
model = MTLModel(config)
model.bart.model.encoder.load_state_dict(load_file(os.path.join(output_dir, "encoder.safetensors")))
model.bart.model.decoder.load_state_dict(load_file(os.path.join(output_dir, "decoder.safetensors")))
model.bart.model.shared.load_state_dict(load_file(os.path.join(output_dir, "shared.safetensors")))
model.bart.lm_head.load_state_dict(load_file(os.path.join(output_dir, "lm_head.safetensors")))
model.classifier_1.load_state_dict(load_file(os.path.join(output_dir, "classifier_1.safetensors")))
model.classifier_2.load_state_dict(load_file(os.path.join(output_dir, "classifier_2.safetensors")))
# Load the tokenizer
tokenizer = BartTokenizer.from_pretrained(output_dir)

# Now you can use `model` and `tokenizer` as needed
model.to(device)
model.train()
from transformers import Trainer
from safetensors.torch import save_file
class CustomMTLTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        # Extract necessary inputs from the batch
        input_ids = inputs['input_ids']
        attention_mask = inputs['attention_mask']
        labels = inputs.get('labels', None)
        classification_input_ids_task_1 = inputs.get('classification_input_ids_task_1', None)
        classification_attention_mask_task_1 = inputs.get('classification_attention_mask_task_1', None)
        classification_labels_task_1 = inputs.get('classification_labels_task_1', None)
        classification_input_ids_task_2 = inputs.get('classification_input_ids_task_2', None)
        classification_attention_mask_task_2 = inputs.get('classification_attention_mask_task_2', None)
        classification_labels_task_2 = inputs.get('classification_labels_task_2', None)
        
        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            classification_input_ids_task_1=classification_input_ids_task_1,
            classification_attention_mask_task_1=classification_attention_mask_task_1,
            classification_labels_task_1=classification_labels_task_1,
            classification_input_ids_task_2=classification_input_ids_task_2,
            classification_attention_mask_task_2=classification_attention_mask_task_2,
            classification_labels_task_2=classification_labels_task_2,
        )
        
        # Get the loss
        loss = outputs[0]  # Assuming your model's forward method returns the loss as the first output
        
        # Ensure the loss is a floating-point type
        loss = loss.float().mean()  # Convert loss to float and calculate the mean
        
        # Update loss weights if necessary
        model.update_loss_weights()
        
        return (loss, outputs) if return_outputs else loss

    def save_model(self, output_dir=None, _internal_call=False):
        # If output_dir is None, use self.args.output_dir
        if output_dir is None:
            output_dir = self.args.output_dir
        
        os.makedirs(output_dir, exist_ok=True)

        # Save each component of the model separately
        save_file(self.model.bart.model.encoder.state_dict(), os.path.join(output_dir, "encoder.safetensors"))
        save_file(self.model.bart.model.decoder.state_dict(), os.path.join(output_dir, "decoder.safetensors"))
        save_file(self.model.bart.model.shared.state_dict(), os.path.join(output_dir, "shared.safetensors"))
        save_file(self.model.bart.lm_head.state_dict(), os.path.join(output_dir, "lm_head.safetensors"))
        
        # Save the classifier separately
        save_file(self.model.classifier_1.state_dict(), os.path.join(output_dir, "classifier_1.safetensors"))
        save_file(self.model.classifier_2.state_dict(), os.path.join(output_dir, "classifier_2.safetensors"))


        # Save tokenizer, config, and other necessary files
        self.tokenizer.save_pretrained(output_dir)
        self.model.bart.config.save_pretrained(output_dir)



training_args = TrainingArguments(
    output_dir='./results-mtl-run',  # Specify where to save checkpoints and outputs
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy='epoch',
    save_steps=1000,
    save_total_limit=2
)

# Initialize the custom trainer
trainer = CustomMTLTrainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=validation,
    tokenizer=tokenizer  # Include tokenizer if you need it for evaluation
)

# Train the model
trainer.train()   

/home/hmoradis/.conda/envs/QA_env/lib/python3.11/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: julien-serbanescu (juteam). Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.603700,2343.760742
2,0.567100,2381.707764
3,0.460900,2371.025391


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file

TrainOutput(global_step=12285, training_loss=0.5173725520298158, metrics={'train_runtime': 5183.616, 'train_samples_per_second': 9.48, 'train_steps_per_second': 2.37, 'total_flos': 0.0, 'train_loss': 0.5173725520298158, 'epoch': 3.0})

In [10]:
output_dir = './fine-tuned-bartMTL-salience'
os.makedirs(output_dir, exist_ok=True)

# Save each component of the model separately
save_file(model.bart.model.encoder.state_dict(), os.path.join(output_dir, "encoder.safetensors"))
save_file(model.bart.model.decoder.state_dict(), os.path.join(output_dir, "decoder.safetensors"))
save_file(model.bart.model.shared.state_dict(), os.path.join(output_dir, "shared.safetensors"))
save_file(model.bart.lm_head.state_dict(), os.path.join(output_dir, "lm_head.safetensors"))

# Save the classifier separately
save_file(model.classifier_1.state_dict(), os.path.join(output_dir, "classifier_1.safetensors"))
save_file(model.classifier_2.state_dict(), os.path.join(output_dir, "classifier_2.safetensors"))

# Save tokenizer and config
tokenizer.save_pretrained(output_dir)
model.bart.config.save_pretrained(output_dir)

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0, 'forced_eos_token_id': 2}
